<div style="border: 2px solid #ff9900; border-radius: 8px; padding: 15px; background-color: #fff3e0; margin-bottom: 10px;">
<strong>⚠️ Compatibility Notice:</strong> This notebook has been tested using <strong>SageMaker Distribution Image 3.7.0</strong> and the <strong>SageMaker Python SDK version 3.4.0</strong>.
</div>

In [ ]:
!pip install -q -U "sagemaker==3.4.0"

In [ ]:
# Restart kernel to pick up the updated packages
import IPython
IPython.Application.instance().kernel.do_shutdown(True)

In [ ]:
import sagemaker
import sagemaker.core
import sagemaker.train
import sagemaker.serve
import sagemaker.mlops
from importlib.metadata import version

print(f"sagemaker: {version('sagemaker')}")
print(f"sagemaker.core: {version('sagemaker.core')}")
print(f"sagemaker.train: {version('sagemaker.train')}")
print(f"sagemaker.server: {version('sagemaker.serve')}")

# SageMaker V3 JumpStart Model Example

This notebook demonstrates how to use SageMaker V3 ModelBuilder with JumpStart models for easy model deployment and inference.

### Prerequisites
Note: Ensure you have sagemaker and ipywidgets installed in your environment. The ipywidgets package is required to monitor endpoint deployment progress in Jupyter notebooks.

In [ ]:
# Import required libraries
import json
import uuid
import boto3

from sagemaker.serve.model_builder import ModelBuilder
from sagemaker.core.jumpstart.configs import JumpStartConfig
from sagemaker.core.resources import EndpointConfig
from sagemaker.train.configs import Compute

## Step 1: Configure JumpStart Model

We'll use a HuggingFace Falcon model from JumpStart for this example.

In [ ]:
# Configuration
# MODEL_ID = "huggingface-llm-falcon-7b-bf16"
MODEL_ID= "meta-textgeneration-llama-3-1-8b-instruct"
MODEL_NAME_PREFIX = "js-v3-example-model"
ENDPOINT_NAME_PREFIX = "js-v3-example-endpoint"

# Generate unique identifiers
unique_id = str(uuid.uuid4())[:8]
model_name = f"{MODEL_NAME_PREFIX}-{unique_id}"
endpoint_name = f"{ENDPOINT_NAME_PREFIX}-{unique_id}"

print(f"Model name: {model_name}")
print(f"Endpoint name: {endpoint_name}")

## Step 2: Create ModelBuilder from JumpStart Config

The ModelBuilder can automatically configure itself from a JumpStart model ID.

In [ ]:
# Initialize model_builder object with JumpStart configuration
compute = Compute(instance_type="ml.g5.2xlarge")
jumpstart_config = JumpStartConfig(model_id=MODEL_ID)
model_builder = ModelBuilder.from_jumpstart_config(jumpstart_config=jumpstart_config, compute=compute)

print("ModelBuilder created successfully from JumpStart config!")

## Step 3: Build the Model

Build the model artifacts and prepare for deployment.

In [ ]:
# Build the model
core_model = model_builder.build(model_name=model_name)
print(f"Model Successfully Created: {core_model.model_name}")

## Step 4: Deploy the Model

Deploy the model to a SageMaker endpoint for real-time inference.

In [ ]:
# Deploy the model to an endpoint
core_endpoint = model_builder.deploy(endpoint_name=endpoint_name)
print(f"Endpoint Successfully Created: {core_endpoint.endpoint_name}")

## Step 5: Test the Endpoint

Send a test request to the deployed endpoint.

In [ ]:
# Test the endpoint with a sample query
test_data = {"inputs": "What are falcons?", "parameters": {"max_new_tokens": 32}}

result = core_endpoint.invoke(
    body=json.dumps(test_data),
    content_type="application/json"
)

# Decode and display the result
prediction = json.loads(result.body.read().decode('utf-8'))
print(f"Model Response: {prediction}")

## Step 5b: Document Summarization with LangChain

Use LangChain's map-reduce chain to summarize a long document by splitting it into chunks, summarizing each chunk, then combining the summaries.


In [ ]:
!pip install -q langchain langchain-aws langchain-community langchain-text-splitters

In [ ]:
with open("document.txt") as f:
    text_to_summarize = f.read()
print(f"Document length: {len(text_to_summarize)} characters")

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain.docstore.document import Document

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=2500, chunk_overlap=20, separators=[" "], length_function=len
)
input_documents = text_splitter.create_documents([text_to_summarize])
print(f"Split into {len(input_documents)} chunks")

In [ ]:
from langchain_aws import SagemakerEndpoint
from langchain_aws.llms.sagemaker_endpoint import LLMContentHandler
from langchain.chains.summarize import load_summarize_chain
from langchain.prompts import PromptTemplate

class FalconContentHandler(LLMContentHandler):
    content_type = "application/json"
    accepts = "application/json"

    def transform_input(self, prompt: str, model_kwargs={}) -> bytes:
        payload = {"inputs": prompt, "parameters": {"max_new_tokens": 1024, "temperature": 0.3, "do_sample": True}}
        return json.dumps(payload).encode("utf-8")

    def transform_output(self, output: bytes) -> str:
        response_json = json.loads(output.read().decode("utf-8"))
        return response_json[0]["generated_text"]

content_handler = FalconContentHandler()

In [ ]:
map_prompt = PromptTemplate(
    template="Write a concise summary of this text in a few complete sentences:\n\n{text}\n\nConcise summary:",
    input_variables=["text"]
)

combine_prompt = PromptTemplate(
    template="Combine all these summaries into a final summary in a few complete sentences:\n\n{text}\n\nFinal summary:",
    input_variables=["text"]
)

In [ ]:
summary_model = SagemakerEndpoint(
    endpoint_name=core_endpoint.endpoint_name,
    region_name=boto3.Session().region_name,
    content_handler=content_handler,
)

summary_chain = load_summarize_chain(
    llm=summary_model,
    chain_type="map_reduce",
    map_prompt=map_prompt,
    combine_prompt=combine_prompt,
    verbose=True,
)

import boto3
result = summary_chain.invoke({"input_documents": input_documents, "token_max": 2048})
print("\n=== Final Summary ===")
print(result["output_text"])

## Step 6: Clean Up Resources

Clean up the created resources to avoid ongoing charges.

In [ ]:
# Clean up resources
core_endpoint_config = EndpointConfig.get(endpoint_config_name=core_endpoint.endpoint_name)

# Delete in the correct order
core_model.delete()
core_endpoint.delete()
core_endpoint_config.delete()

print("All resources successfully deleted!")

## Summary

This notebook demonstrated:
1. Creating a ModelBuilder from JumpStart configuration
2. Building a model from JumpStart
3. Deploying to a SageMaker endpoint
4. Making inference requests
5. Cleaning up resources

The V3 ModelBuilder makes it easy to work with JumpStart models with minimal configuration required!